# 配对样本假设检验完整教程：从手算到 scipy 验证

## 📚 教学目标
1. 理解配对样本和独立样本的区别
2. 掌握配对 t 检验的完整计算步骤
3. 计算差值的均值 $ar{d}$ 和标准差 $s_d$
4. 用 `scipy.stats.ttest_rel` 验证手算结果
5. 理解配对设计消除个体变异、提高检验效能的优势

**参考书**：《基础统计学(第14版)》（Triola）第 9-3 节

**教学策略**：先用 8 对微型数据集让你「看见」每一步手算，再用 scipy 验证，最后扩展到 200 对大样本

## 1. 场景设定：为什么要用配对设计？

### 🎯 问题

在医学、心理学、教育学等领域，我们经常需要比较**同一个体**在两种条件下的表现，或者**精心匹配**的两个受试组之间的差异。

**典型场景：**
- 🏥 **前后测量**：同一批病人治疗前 vs 治疗后的血压
- ⚖️ **体重研究**：同一个人的自报体重 vs 实测体重
- 🧪 **药物试验**：同一受试者的左手 vs 右手反应时间
- 📚 **教学实验**：同一组学生的前测成绩 vs 后测成绩

### 📖 书中的定义

> **配对样本**（Paired Sample）：两个数据集之间存在自然的逐对对应关系。配对 t 检验通过分析**每对数据的差值** $d_i$ 来检验两个总体均值是否相等。

配对设计的核心思想：**用差值消除个体间变异**，让真实的处理效应更容易被检测到。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

## 2. 独立样本 vs 配对样本

理解两者的区别是正确选择检验方法的前提。

### 📝 关键区别对比

| 特性 | 独立样本 (Independent) | 配对样本 (Paired) |
|------|----------------------|------------------|
| **数据关系** | 两组互不相关 | 两组存在逐对对应 |
| **样本量** | $n_1$, $n_2$ 可以不同 | 两个样本量必须相等 $n$ |
| **分析对象** | 直接比较两组均值 | 分析**每对的差值** $d_i$ |
| **变异来源** | 包含个体间变异 | ✅ 消除了个体间变异 |
| **检验效能** | 较低（噪声大） | ✅ 较高（噪声小） |
| **适用场景** | 不同个体的两组 | 同一受试者的两次测量 |
| **scipy 函数** | `ttest_ind` | `ttest_rel` |

### 💡 核心洞察

配对设计的**核心优势**：当个体之间差异很大（如体重从 120 斤到 240 斤），但处理效应很小（如自报与实测只差几斤）时，
配对设计通过计算**每个人自己的差值**，把巨大的个体差异「减掉」了，
让微小的处理效应在剩余的差值变异中更容易显现。

In [ ]:
# ========== 直观对比：独立 vs 配对 ==========

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 模拟数据 ---
n_subjects = 30
true_weight = np.random.normal(170, 30, n_subjects)
bias = -3  # 自报比实测轻 3 磅
reported = true_weight + bias + np.random.normal(0, 4, n_subjects)

# --- Plot 1: Independent samples (ignoring pairing) ---
ax1 = axes[0]
positions = [1, 2]
bp1 = ax1.boxplot([true_weight, reported], positions=positions,
                  labels=['Measured Weight', 'Reported Weight'],
                  patch_artist=True, widths=0.5)
bp1['boxes'][0].set_facecolor('#2ecc71')
bp1['boxes'][1].set_facecolor('#e74c3c')
for patch in bp1['boxes']:
    patch.set_alpha(0.7)
ax1.set_ylabel('Weight (lbs)', fontsize=12)
ax1.set_title('Independent View: Two Groups', fontsize=14, fontweight='bold')
ax1.set_xlim(0.3, 2.7)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: Paired view (connected pairs) ---
ax2 = axes[1]
for i in range(n_subjects):
    color = '#2ecc71' if reported[i] < true_weight[i] else '#e74c3c'
    ax2.plot([1, 2], [true_weight[i], reported[i]], 'o-',
             color=color, alpha=0.5, markersize=5)
ax2.set_xticks([1, 2])
ax2.set_xticklabels(['Measured Weight', 'Reported Weight'], fontsize=10)
ax2.set_ylabel('Weight (lbs)', fontsize=12)
ax2.set_title('Paired View: Connected Pairs', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- Plot 3: Differences (paired analysis) ---
ax3 = axes[2]
differences = reported - true_weight
colors = ['#2ecc71' if d < 0 else '#e74c3c' for d in differences]
ax3.bar(range(n_subjects), differences, color=colors, alpha=0.7, edgecolor='white')
ax3.axhline(y=0, color='black', linewidth=1)
ax3.axhline(y=np.mean(differences), color='steelblue', linestyle='--',
            linewidth=2, label=f'Mean d = {np.mean(differences):.2f}')
ax3.set_xlabel('Subject Index', fontsize=12)
ax3.set_ylabel('Difference (Reported - Measured)', fontsize=12)
ax3.set_title('Difference Distribution', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1（左）：独立视角 — 两组数据重叠严重，看不出差异")
print(f"  图2（中）：配对视角 — 每条线连接同一个人的两次测量")
print(f"  图3（右）：差值分布 — 大部分为负值（自报偏低），差异清晰可见")
print(f"\n  ✅ 配对设计把隐藏在巨大个体差异中的微小系统偏差暴露了出来！")

## 3. 配对 t 检验的适用条件

### 📐 检验前提

使用配对 t 检验前，需要确认以下三个条件：

1. **样本数据为配对样本** — 两个数据集之间存在自然的逐对对应关系
   - 例：同一受试者的前后测量、匹配的受试对

2. **配对样本为简单随机样本** — 每对观测是独立抽取的
   - 不能是方便抽样或自愿响应样本

3. **差值总体服从正态分布，或 $n > 30$
   - 小样本时：需要 Shapiro-Wilk 检验确认差值的正态性
   - 大样本时：中心极限定理保证 $\bar{d}$ 的近似正态性

### 📐 检验假设

$$H_0: \mu_d = 0 \quad \text{(两个条件的总体均值无差异)}$$
$$H_1: \mu_d \neq 0 \quad \text{或} \quad \mu_d > 0 \quad \text{或} \quad \mu_d < 0$$

其中 $\mu_d$ 是差值的总体均值。

### 📐 检验统计量

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

其中：
- $\bar{d}$ = 差值的样本均值
- $s_d$ = 差值的样本标准差
- $n$ = 配对数量
- $df$ = 自由度

## 4. 微型数据手算：实测体重 vs 自报体重

### 📖 教材数据

来自《基础统计学》第 9-3 节的经典例题：8 名受试者的实测体重与自报体重（单位：磅）。

### 📐 差值计算公式

$$d_i = x_i - y_i$$

其中：
- $x_i$ = 第 $i$ 个受试者的实测体重
- $y_i$ = 第 $i$ 个受试者的自报体重
- $d_i$ = 差值（实测 − 自报）

### 📐 差值的均值

$$\bar{d} = \frac{\sum d_i}{n}$$

### 📐 差值的标准差

$$s_d = \sqrt{\frac{\sum(d_i - \bar{d})^2}{n - 1}}$$

In [ ]:
# ========== 原始数据 ==========
# 📖 教材数据：8 名受试者的体重（磅）
measured  = np.array([152.6, 149.3, 174.8, 119.5, 194.9, 180.3, 215.4, 239.6])
reported  = np.array([150.0, 148.0, 170.0, 119.0, 185.0, 180.0, 224.0, 239.0])
n = len(measured)

df_weight = pd.DataFrame({
    'Subject': [f'S{i+1}' for i in range(n)],
    'Reported_lbs': reported,
    'Measured_lbs': measured
})

print("📊 原始数据：")
print(df_weight.to_string(index=False))

# ========== 步骤 1: 计算每对差值 ==========
print(f"\n{'='*60}")
print(f"📊 步骤 1: 计算每对差值 d = 实测 - 自报")
print(f"{'='*60}")

d = measured - reported
df_weight['Difference_d'] = d

print(f"\n  {'受试者':<8} {'实测':>8} {'自报':>8} {'差值 d':>8}")
print(f"  {'-'*36}")
for i in range(n):
    print(f"  {'S'+str(i+1):<8} {measured[i]:>8.1f} {reported[i]:>8.1f} {d[i]:>8.1f}")

# ========== 步骤 2: 计算差值的均值 ==========
print(f"\n{'='*60}")
print(f"📊 步骤 2: 计算差值的均值 d̄")
print(f"{'='*60}")

d_bar = np.sum(d) / n
print(f"  计算公式: d̄ = Σdᵢ / n")
print(f"  Σdᵢ = {' + '.join([f'{x:.1f}' for x in d])} = {np.sum(d):.1f}")
print(f"  n = {n}")
print(f"  d̄ = {np.sum(d):.1f} / {n} = {d_bar:.4f}")
print(f"  np.mean 验证: {np.mean(d):.4f}")
print(f"\n  💡 解读: 自报体重平均比实测体重轻 {d_bar:.4f} 磅")
print(f"  （d̄ > 0 表示实测高于自报，即人们倾向于少报体重）")

### 📐 差值标准差的计算

$$s_d = \sqrt{\frac{\sum(d_i - \bar{d})^2}{n - 1}}$$

其中：
- $d_i$ = 第 $i$ 对的差值
- $\bar{d}$ = 差值的均值
- $n$ = 配对数量
- $n - 1$ = 自由度

In [ ]:
# ========== 步骤 3: 计算差值的标准差 ==========
print(f"{'='*60}")
print(f"📊 步骤 3: 计算差值的标准差 s_d")
print(f"{'='*60}")

deviations = d - d_bar
ss = np.sum(deviations ** 2)
variance_d = ss / (n - 1)
s_d = np.sqrt(variance_d)

print(f"\n  每个差值与均值的偏差:")
print(f"  {'受试者':<8} {'dᵢ':>8} {'dᵢ - d̄':>10} {'(dᵢ - d̄)²':>12}")
print(f"  {'-'*42}")
for i in range(n):
    print(f"  {'S'+str(i+1):<8} {d[i]:>8.1f} {deviations[i]:>10.4f} {deviations[i]**2:>12.4f}")

print(f"\n  Σ(dᵢ - d̄)² = {ss:.4f}")
print(f"  s² = Σ(dᵢ - d̄)² / (n-1) = {ss:.4f} / {n-1} = {variance_d:.4f}")
print(f"  s_d = √{variance_d:.4f} = {s_d:.6f}")
print(f"  np.std(ddof=1) 验证: {np.std(d, ddof=1):.6f}")
print(f"\n  💡 解读: 差值的标准差为 {s_d:.4f}，说明个体间的自报偏差差异较大")

### 📐 检验统计量的计算

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}$$

其中：
- $\bar{d}$ = 差值的均值 = 1.425
- $s_d$ = 差值的标准差
- $n$ = 配对数量 = 8
- $s_d / \sqrt{n}$ = 差值均值的标准误 (SE)

**检验设置**（右侧检验）：
$$H_0: \mu_d = 0$$
$$H_1: \mu_d > 0$$
$$\alpha = 0.05$$

In [ ]:
# ========== 步骤 4: 计算检验统计量 ==========
print(f"{'='*60}")
print(f"📊 步骤 4: 计算检验统计量 t")
print(f"{'='*60}")

se = s_d / np.sqrt(n)
t_stat = d_bar / se
df = n - 1

print(f"  标准误 SE = s_d / √n = {s_d:.4f} / √{n} = {se:.4f}")
print(f"  t = d̄ / SE = {d_bar:.4f} / {se:.4f} = {t_stat:.4f}")
print(f"  自由度 df = n - 1 = {n} - 1 = {df}")

# ========== 步骤 5: 计算 p 值 ==========
print(f"\n{'='*60}")
print(f"📊 步骤 5: 计算 p 值（右侧检验）")
print(f"{'='*60}")

# 右侧检验: P(T > t_stat)
p_value_right = 1 - stats.t.cdf(t_stat, df=df)
# 双侧检验供参考
p_value_two = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))

print(f"  右侧检验 p = P(T > {t_stat:.4f}) = {p_value_right:.6f}")
print(f"  双侧检验 p = 2 × P(T > |{t_stat:.4f}|) = {p_value_two:.6f}")

# ========== 步骤 6: 做出决策 ==========
alpha = 0.05
print(f"\n{'='*60}")
print(f"📊 步骤 6: 做出决策")
print(f"{'='*60}")

t_critical = stats.t.ppf(1 - alpha, df=df)
print(f"  显著性水平 α = {alpha}")
print(f"  临界值 t_α = {t_critical:.4f}")
print(f"  检验统计量 t = {t_stat:.4f}")
print(f"  p 值 = {p_value_right:.6f}")

if p_value_right < alpha:
    print(f"\n  ✗ 结论: p = {p_value_right:.4f} < α = {alpha}")
    print(f"  ✗ 拒绝 H₀：自报体重显著高于实测体重")
else:
    print(f"\n  ✓ 结论: p = {p_value_right:.4f} ≥ α = {alpha}")
    print(f"  ✓ 不拒绝 H₀：没有足够证据证明自报体重高于实测体重")
    print(f"  ✓ 教材结论: p > 0.10（与书中一致）")

## 5. 使用 scipy.stats.ttest_rel 验证

### 🔬 验证模式

手算完成后，我们用 scipy 的内置函数进行交叉验证，确保计算正确。

> **注意**：`scipy.stats.ttest_rel` 默认进行**双侧检验**，我们需要手动转换为单侧 p 值。

In [ ]:
# ========== 用 scipy 验证 ==========
print("{" + "="*58 + "}")
print("🔬 scipy.stats.ttest_rel 验证")
print("{" + "="*58 + "}")

# scipy 配对 t 检验
t_scipy, p_scipy_two = stats.ttest_rel(measured, reported)

# 转换为单侧 p 值（右侧检验: μ_d > 0）
if t_scipy > 0:
    p_scipy_right = p_scipy_two / 2  # 右侧
else:
    p_scipy_right = 1 - p_scipy_two / 2

print(f"\n  📊 手算 vs scipy 对比:")
print(f"  {'指标':<15} {'手算':>12} {'scipy':>12} {'差异':>12}")
print(f"  {'-'*51}")
print(f"  {'T 统计量':<15} {t_stat:>12.6f} {t_scipy:>12.6f} {abs(t_stat-t_scipy):>12.8f}")
print(f"  {'双侧 p 值':<15} {p_value_two:>12.6f} {p_scipy_two:>12.6f} {abs(p_value_two-p_scipy_two):>12.8f}")
print(f"  {'单侧 p 值':<15} {p_value_right:>12.6f} {p_scipy_right:>12.6f} {abs(p_value_right-p_scipy_right):>12.8f}")

tolerance = 1e-10
if abs(t_stat - t_scipy) < tolerance and abs(p_value_two - p_scipy_two) < tolerance:
    print(f"\n  ✅ 验证通过！手算结果与 scipy 完全一致")
else:
    print(f"\n  ⚠️ 存在微小差异，可能是浮点精度导致")

# 打印 ttest_rel 的详细结果
print(f"\n  📋 scipy.stats.ttest_rel 返回值:")
print(f"     statistic = {t_scipy:.6f}")
print(f"     pvalue    = {p_scipy_two:.6f} (双侧)")
print(f"     df        = {n - 1}")

## 6. 置信区间构建

### 📐 配对差值的置信区间公式

$$\bar{d} - E < \mu_d < \bar{d} + E$$

$$E = t_{\alpha/2} \times \frac{s_d}{\sqrt{n}}$$

其中：
- $\bar{d}$ = 差值的样本均值
- $t_{\alpha/2}$ = 置信水平对应的 t 临界值
- $s_d$ = 差值的标准差
- $n$ = 配对数量
- $E$ = 误差范围（Margin of Error）

In [ ]:
# ========== 步骤 7: 构建置信区间 ==========
confidence_level = 0.95
alpha_ci = 1 - confidence_level
t_critical_ci = stats.t.ppf(1 - alpha_ci/2, df=df)
margin_of_error = t_critical_ci * se
ci_lower = d_bar - margin_of_error
ci_upper = d_bar + margin_of_error

print(f"{'='*60}")
print(f"📊 步骤 7: 构建 {confidence_level*100:.0f}% 置信区间")
print(f"{'='*60}")
print(f"\n  计算公式: d̄ ± t_(α/2) × (s_d / √n)")
print(f"  t_(α/2) = t_{alpha_ci/2:.3f}({df}) = {t_critical_ci:.4f}")
print(f"  误差范围 E = {t_critical_ci:.4f} × {se:.4f} = {margin_of_error:.4f}")
print(f"  置信区间 = {d_bar:.4f} ± {margin_of_error:.4f}")
print(f"  置信区间 = ({ci_lower:.4f}, {ci_upper:.4f})")

print(f"\n  💡 解读:")
print(f"  我们有 {confidence_level*100:.0f}% 的信心认为，自报与实测体重差值的")
print(f"  真实总体均值 μ_d 在 ({ci_lower:.4f}, {ci_upper:.4f}) 之间")
if ci_lower < 0 < ci_upper:
    print(f"  ⚠️ 注意：区间包含 0，与不拒绝 H₀ 的结论一致")
else:
    print(f"  ⚠️ 注意：区间不包含 0，与拒绝 H₀ 的结论一致")

## 7. 可视化

### 🎨 三幅图全面展示配对分析结果

In [ ]:
# ========== 可视化 1: 差值分布 + t 分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1a: 差值柱状图 ---
ax1 = axes[0]
colors_bar = ['#2ecc71' if di > 0 else '#e74c3c' for di in d]
ax1.bar(range(n), d, color=colors_bar, alpha=0.7, edgecolor='white', linewidth=1.5)
ax1.axhline(y=0, color='black', linewidth=1)
ax1.axhline(y=d_bar, color='steelblue', linestyle='--', linewidth=2,
            label=f'Mean d = {d_bar:.2f}')
ax1.set_xlabel('Subject Index', fontsize=12)
ax1.set_ylabel('Difference (Measured - Reported)', fontsize=12)
ax1.set_title('Paired Differences per Subject', fontsize=14, fontweight='bold')
ax1.set_xticks(range(n))
ax1.set_xticklabels([f'S{i+1}' for i in range(n)], fontsize=9)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 1b: t 分布与检验统计量 ---
ax2 = axes[1]
x_t = np.linspace(-4, 4, 1000)
y_t = stats.t.pdf(x_t, df=df)
ax2.plot(x_t, y_t, 'k-', linewidth=2, label=f't distribution (df={df})')
ax2.fill_between(x_t, 0, y_t, alpha=0.1, color='steelblue')

# 标记检验统计量
ax2.axvline(x=t_stat, color='#e74c3c', linestyle='-', linewidth=2,
            label=f't = {t_stat:.4f}')
# 标记临界值
ax2.axvline(x=t_critical, color='#e67e22', linestyle='--', linewidth=1.5,
            label=f't_critical = {t_critical:.4f}')
# 填充拒绝域
x_reject = np.linspace(t_critical, 4, 500)
y_reject = stats.t.pdf(x_reject, df=df)
ax2.fill_between(x_reject, 0, y_reject, alpha=0.3, color='#e74c3c',
                 label='Rejection Region')

ax2.set_xlabel('T Value', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('T Distribution and Test Statistic', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1（左）：差值柱状图 — 绿色=实测高于自报（少报），红色=实测低于自报（多报）")
print(f"  图2（右）：t 分布 — 红色线 t={t_stat:.4f} 未落入拒绝域")
print(f"  橙色虚线为临界值 t={t_critical:.4f}（α=0.05, df={df}）")

In [ ]:
# ========== 可视化 2: 置信区间图 ==========
fig, ax = plt.subplots(figsize=(10, 5))

# 绘制置信区间
ax.barh(0, ci_upper - ci_lower, left=ci_lower, height=0.4,
        color='steelblue', alpha=0.3, edgecolor='steelblue', linewidth=2,
        label=f'{confidence_level*100:.0f}% CI')
# 标记均值
ax.plot(d_bar, 0, 'D', color='steelblue', markersize=12, zorder=5,
        label=f'Mean d = {d_bar:.4f}')
# 标记零线
ax.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8,
           label='H₀: μ_d = 0')
# 标记区间边界
ax.plot([ci_lower, ci_upper], [0, 0], '|', color='steelblue',
        markersize=20, markeredgewidth=2)

ax.set_xlabel('Difference Value (lbs)', fontsize=12)
ax.set_title(f'{confidence_level*100:.0f}% Confidence Interval for μ_d',
             fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.legend(fontsize=10, loc='upper right')
ax.grid(axis='x', alpha=0.3)
ax.set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色条带 = {confidence_level*100:.0f}% 置信区间 ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"  蓝色菱形 = 差值均值 d̄ = {d_bar:.4f}")
print(f"  红色虚线 = 零假设 H₀: μ_d = 0")
if ci_lower < 0 < ci_upper:
    print(f"  置信区间包含 0，不能拒绝 H₀")
else:
    print(f"  置信区间不包含 0，拒绝 H₀")

In [ ]:
# ========== 可视化 3: 配对连接图 ==========
fig, ax = plt.subplots(figsize=(8, 6))

x_pos = [1, 2]
for i in range(n):
    color = '#2ecc71' if d[i] > 0 else '#e74c3c'
    ax.plot(x_pos, [measured[i], reported[i]], 'o-',
            color=color, alpha=0.6, markersize=8, linewidth=1.5)

# 均值连线
ax.plot(x_pos, [np.mean(measured), np.mean(reported)], 's-',
        color='steelblue', markersize=12, linewidth=3,
        label=f'Means: {np.mean(measured):.1f} → {np.mean(reported):.1f}',
        zorder=5)

ax.set_xticks(x_pos)
ax.set_xticklabels(['Measured Weight', 'Reported Weight'], fontsize=12)
ax.set_ylabel('Weight (lbs)', fontsize=12)
ax.set_title('Before-After Connected Plot: Measured vs Reported',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色线：实测 > 自报（少报了自己的体重）")
print(f"  红色线：实测 < 自报（多报了自己的体重）")
print(f"  蓝色粗线：两组均值的连线")
print(f"  每条线代表同一个受试者的两次测量——这就是「配对」的含义")

## 8. 配对设计的优势：消除个体变异

### 💡 为什么配对设计更好？

配对设计的核心优势是**消除个体间变异**。让我们通过模拟来量化这个优势。

**思想实验**：如果我们错误地把这 8 对数据当作独立样本来分析，会怎样？

In [ ]:
# ========== 配对 vs 独立：效能对比 ==========
print(f"{'='*60}")
print(f"💡 配对设计 vs 独立设计的效能对比")
print(f"{'='*60}")

# --- 方法 1: 配对 t 检验（正确方法）---
t_paired, p_paired = stats.ttest_rel(measured, reported)
print(f"\n  📊 方法 1: 配对 t 检验（正确）")
print(f"  分析对象: 差值 d = 实测 - 自报")
print(f"  d̄ = {np.mean(d):.4f}")
print(f"  s_d = {np.std(d, ddof=1):.4f}")
print(f"  t = {t_paired:.4f}")
print(f"  p = {p_paired:.4f} (双侧)")

# --- 方法 2: 独立样本 t 检验（错误方法）---
t_ind, p_ind = stats.ttest_ind(reported, measured)
print(f"\n  📊 方法 2: 独立样本 t 检验（忽略配对）")
print(f"  分析对象: 直接比较两组均值")
print(f"  均值1 (reported) = {np.mean(reported):.4f}")
print(f"  均值2 (measured) = {np.mean(measured):.4f}")
print(f"  t = {t_ind:.4f}")
print(f"  p = {p_ind:.4f} (双侧)")

# --- 方差分解 ---
print(f"\n  📊 方差分解:")
print(f"  差值标准差 s_d    = {np.std(d, ddof=1):.4f}（配对分析使用）")
print(f"  实测组标准差      = {np.std(measured, ddof=1):.4f}")
print(f"  自报组标准差      = {np.std(reported, ddof=1):.4f}")
print(f"  合并标准差        = {np.sqrt((np.var(measured, ddof=1) + np.var(reported, ddof=1))/2):.4f}（独立分析使用）")
print(f"\n  💡 差值的标准差远小于原始数据的标准差")
print(f"     → 更小的标准误 → 更大的 t 值 → 更容易检测到差异")

# --- 可视化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：原始数据分布
ax1 = axes[0]
ax1.hist(measured, bins=8, alpha=0.5, color='#2ecc71', edgecolor='white',
         label='Measured', density=True)
ax1.hist(reported, bins=8, alpha=0.5, color='#e74c3c', edgecolor='white',
         label='Reported', density=True)
ax1.set_xlabel('Weight (lbs)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Raw Data: High Variability', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 右图：差值分布
ax2 = axes[1]
ax2.hist(d, bins=6, alpha=0.7, color='steelblue', edgecolor='white',
         density=True)
x_norm = np.linspace(min(d)-2, max(d)+2, 100)
y_norm = stats.norm.pdf(x_norm, np.mean(d), np.std(d, ddof=1))
ax2.plot(x_norm, y_norm, 'r-', linewidth=2, label='Normal Fit')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='H₀: d = 0')
ax2.set_xlabel('Difference (lbs)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Differences: Low Variability', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n  ✅ 关键洞察：")
print(f"  配对设计通过「减去个体差异」，把数据的变异大幅缩小")
print(f"  原始数据范围: [{min(measured):.1f}, {max(measured):.1f}]，跨度 = {max(measured)-min(measured):.1f}")
print(f"  差值范围: [{min(d):.1f}, {max(d):.1f}]，跨度 = {max(d)-min(d):.1f}")
print(f"  变异缩减了 {(1 - (max(d)-min(d))/(max(measured)-min(measured)))*100:.1f}%！")

## 9. 大样本扩展：200 对配对数据

从 8 对微型数据扩展到 200 对，展示大样本下的配对 t 检验流程。

### 📐 数据生成过程 (DGP)

- 真实体重：$Y_i \sim N(170, 30^2)$
- 自报偏差：$\epsilon_i \sim N(-3, 4^2)$（系统性地少报约 3 磅）
- 自报体重：$X_i = Y_i + \epsilon_i$

In [ ]:
# ========== 生成大样本配对数据 ==========

# --- 数据生成过程 (DGP) ---
n_large = 200
true_weight_large = np.random.normal(170, 30, n_large)
# 自报偏差: 系统性少报约 3 磅 + 随机误差
bias_large = np.random.normal(-3, 4, n_large)
reported_large = true_weight_large + bias_large

# 计算差值: d = 实测 - 自报
d_large = true_weight_large - reported_large

print(f"{'='*60}")
print(f"📊 大样本配对数据")
print(f"{'='*60}")
print(f"\n  样本量: n = {n_large}")
print(f"  真实体重: ~ N(170, 30²)")
print(f"  自报偏差: ~ N(-3, 4²)")
print(f"  差值均值 d̄ = {np.mean(d_large):.4f}")
print(f"  差值标准差 s_d = {np.std(d_large, ddof=1):.4f}")

In [ ]:
# ========== 大样本手算 + scipy 验证 ==========
print(f"{'='*60}")
print(f"📊 大样本配对 t 检验")
print(f"{'='*60}")

# --- 手算 ---
d_bar_l = np.mean(d_large)
s_d_l = np.std(d_large, ddof=1)
se_l = s_d_l / np.sqrt(n_large)
t_stat_l = d_bar_l / se_l
df_l = n_large - 1
p_value_l = 2 * (1 - stats.t.cdf(abs(t_stat_l), df=df_l))

print(f"\n  📐 手算步骤:")
print(f"  d̄ = {d_bar_l:.4f}")
print(f"  s_d = {s_d_l:.4f}")
print(f"  SE = {s_d_l:.4f} / √{n_large} = {se_l:.4f}")
print(f"  t = {d_bar_l:.4f} / {se_l:.4f} = {t_stat_l:.4f}")
print(f"  df = {df_l}")
print(f"  p = {p_value_l:.6f}")

# --- scipy 验证 ---
t_scipy_l, p_scipy_l = stats.ttest_rel(true_weight_large, reported_large)

print(f"\n  🔬 scipy 验证:")
print(f"  {'指标':<12} {'手算':>12} {'scipy':>12}")
print(f"  {'-'*38}")
print(f"  {'T':<12} {t_stat_l:>12.6f} {t_scipy_l:>12.6f}")
print(f"  {'p':<12} {p_value_l:>12.6f} {p_scipy_l:>12.6f}")
print(f"  {'df':<12} {df_l:>12} {n_large-1:>12}")

if abs(t_stat_l - t_scipy_l) < 1e-10:
    print(f"\n  ✅ 验证通过！")

# --- 置信区间 ---
alpha_l = 0.05
t_crit_l = stats.t.ppf(1 - alpha_l/2, df=df_l)
e_l = t_crit_l * se_l
ci_l_lower = d_bar_l - e_l
ci_l_upper = d_bar_l + e_l

print(f"\n  📐 95% 置信区间:")
print(f"  E = {t_crit_l:.4f} × {se_l:.4f} = {e_l:.4f}")
print(f"  CI = ({ci_l_lower:.4f}, {ci_l_upper:.4f})")

# --- 效应量 ---
cohens_d_l = d_bar_l / s_d_l
print(f"\n  📐 效应量 Cohen's d:")
print(f"  d = |d̄| / s_d = {abs(d_bar_l):.4f} / {s_d_l:.4f} = {abs(cohens_d_l):.4f}")
if abs(cohens_d_l) < 0.2:
    effect_label = "小"
elif abs(cohens_d_l) < 0.5:
    effect_label = "中等"
elif abs(cohens_d_l) < 0.8:
    effect_label = "大"
else:
    effect_label = "非常大"
print(f"  效应大小: {effect_label}")

print(f"\n  🎯 结论:")
if p_value_l < alpha_l:
    print(f"  p = {p_value_l:.6f} < α = {alpha_l}")
    print(f"  拒绝 H₀：自报体重与实测体重存在显著差异")
    print(f"  自报体重系统性地低于实测体重约 {abs(d_bar_l):.2f} 磅")
else:
    print(f"  p = {p_value_l:.6f} ≥ α = {alpha_l}")
    print(f"  不拒绝 H₀")

In [ ]:
# ========== 大样本可视化 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: 差值直方图 + 正态拟合 ---
ax1 = axes[0]
ax1.hist(d_large, bins=25, density=True, alpha=0.6, color='steelblue',
         edgecolor='white', label='Differences')
x_fit = np.linspace(min(d_large), max(d_large), 200)
y_fit = stats.norm.pdf(x_fit, d_bar_l, s_d_l)
ax1.plot(x_fit, y_fit, 'r-', linewidth=2, label='Normal Fit')
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='H₀: d = 0')
ax1.axvline(x=d_bar_l, color='#2ecc71', linestyle='-', linewidth=2,
            label=f'Mean = {d_bar_l:.2f}')
ax1.set_xlabel('Difference (lbs)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title(f'Difference Distribution (n={n_large})', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: 置信区间图 ---
ax2 = axes[1]
ax2.barh(0, ci_l_upper - ci_l_lower, left=ci_l_lower, height=0.3,
         color='steelblue', alpha=0.3, edgecolor='steelblue', linewidth=2)
ax2.plot(d_bar_l, 0, 'D', color='steelblue', markersize=12, zorder=5)
ax2.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8,
            label='H₀: μ_d = 0')
ax2.plot([ci_l_lower, ci_l_upper], [0, 0], '|', color='steelblue',
         markersize=20, markeredgewidth=2)
ax2.set_xlabel('Difference Value (lbs)', fontsize=12)
ax2.set_title(f'95% CI for μ_d (n={n_large})', fontsize=14, fontweight='bold')
ax2.set_yticks([])
ax2.legend(fontsize=10)
ax2.grid(axis='x', alpha=0.3)
ax2.set_ylim(-0.5, 0.5)

# --- Plot 3: t 分布 ---
ax3 = axes[2]
x_t_l = np.linspace(-8, 8, 1000)
y_t_l = stats.t.pdf(x_t_l, df=df_l)
ax3.plot(x_t_l, y_t_l, 'k-', linewidth=2, label=f't (df={df_l})')
ax3.fill_between(x_t_l, 0, y_t_l, alpha=0.1, color='steelblue')

# 双侧拒绝域
t_crit_abs = stats.t.ppf(1 - alpha_l/2, df=df_l)
x_rej_right = np.linspace(t_crit_abs, 8, 500)
x_rej_left = np.linspace(-8, -t_crit_abs, 500)
ax3.fill_between(x_rej_right, 0, stats.t.pdf(x_rej_right, df_l),
                 alpha=0.3, color='#e74c3c', label='Rejection Region')
ax3.fill_between(x_rej_left, 0, stats.t.pdf(x_rej_left, df_l),
                 alpha=0.3, color='#e74c3c')
ax3.axvline(x=t_stat_l, color='#e74c3c', linestyle='-', linewidth=2,
            label=f't = {t_stat_l:.2f}')
ax3.set_xlabel('T Value', fontsize=12)
ax3.set_ylabel('Probability Density', fontsize=12)
ax3.set_title('Test Statistic Position', fontsize=14, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1（左）：差值分布接近正态，均值明显偏离 0")
print(f"  图2（中）：95% 置信区间不包含 0，差异显著")
print(f"  图3（右）：t 统计量远超出拒绝域临界值")
print(f"  大样本下，配对 t 检验轻松检测到 -3 磅的系统偏差")

## 10. 结果报告

### 📋 标准统计报告格式

In [ ]:
# ========== 完整结果报告 ==========
print("=" * 60)
print("📋 配对样本 t 检验结果报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   自报体重与实测体重是否存在显著差异？")
print(f"   假设：人们倾向于系统性地少报体重")

print(f"\n📊 数据概况:")
print(f"   样本量: n = {n_large} 对")
print(f"   数据来源: 模拟数据（DGP: 自报偏差 ~ N(-3, 4²)）")
print(f"   差值均值 d̄ = {d_bar_l:.4f} 磅")
print(f"   差值标准差 s_d = {s_d_l:.4f} 磅")
print(f"   差值标准误 SE = {se_l:.4f} 磅")

print(f"\n🧮 假设检验:")
print(f"   H₀: μ_d = 0（自报与实测无差异）")
print(f"   H₁: μ_d ≠ 0（存在差异）")
print(f"   显著性水平 α = 0.05")
print(f"\n   检验统计量:")
print(f"   t({df_l}) = {t_stat_l:.4f}")
print(f"   p = {p_value_l:.6f}")

print(f"\n📐 效应量:")
print(f"   Cohen's d = {abs(cohens_d_l):.4f} ({effect_label}效应)")

print(f"\n📐 置信区间:")
print(f"   95% CI = ({ci_l_lower:.4f}, {ci_l_upper:.4f})")
print(f"   不包含 0，与拒绝 H₀ 的结论一致")

print(f"\n🎯 结论:")
if p_value_l < alpha_l:
    print(f"   在 α = 0.05 的显著性水平下，")
    print(f"   配对 t 检验表明自报体重与实测体重存在显著差异，")
    print(f"   t({df_l}) = {t_stat_l:.4f}, p < 0.001, d = {abs(cohens_d_l):.4f}.")
    print(f"   人们系统性地少报体重，平均少报约 {abs(d_bar_l):.2f} 磅。")

print(f"\n💡 配对设计的效果:")
print(f"   差值标准差 ({s_d_l:.4f}) 远小于原始数据标准差")
print(f"   配对设计有效消除了个体间变异，提高了检验效能")

print("\n" + "=" * 60)

## 11. 📌 核心概念回顾

### 📌 配对样本 (Paired Sample)
- **定义**: 两个数据集之间存在自然的逐对对应关系
- **典型场景**: 同一受试者的前后测量、匹配的受试对、左右对照
- **核心优势**: 通过分析差值消除个体间变异，提高检验效能

### 📌 配对 t 检验 (Paired t-Test)
- **定义**: 基于配对样本差值的单样本 t 检验
- **公式**: $t = \bar{d} / (s_d / \sqrt{n})$，$df = n - 1$
- **含义**: 检验差值的总体均值 $\mu_d$ 是否等于某个值（通常为 0）
- **判断标准**: $|t| > t_{\alpha/2}$ 或 $p < \alpha$ 时拒绝 $H_0$

### 📌 差值的均值与标准差
- **定义**: 将配对数据转化为差值后，计算差值的描述统计量
- **公式**: $\bar{d} = \sum d_i / n$，$s_d = \sqrt{\sum(d_i - \bar{d})^2 / (n-1)}$
- **意义**: $\bar{d}$ 反映系统偏差的方向和大小，$s_d$ 反映偏差的变异程度

### 📌 置信区间
- **定义**: 对差值总体均值 $\mu_d$ 的区间估计
- **公式**: $\bar{d} \pm t_{\alpha/2} \times (s_d / \sqrt{n})$
- **含义**: 有 95% 的信心认为 $\mu_d$ 落在此区间内
- **判断标准**: 区间不包含 0 等价于在 $\alpha = 0.05$ 水平下拒绝 $H_0$

### 📌 效应量 Cohen's d
- **定义**: 标准化的效应大小指标
- **公式**: $d = |\bar{d}| / s_d$
- **含义**: 消除量纲后，衡量差异的实际大小
- **判断标准**: $d < 0.2$（小）、$0.2 \leq d < 0.5$（中）、$0.5 \leq d < 0.8$（大）、$d \geq 0.8$（非常大）

### 🔗 完整流程
```
收集配对数据 → 计算差值 dᵢ → 计算 d̄ 和 s_d → 计算 t 统计量
      ↓              ↓              ↓              ↓
  确认配对关系    dᵢ = xᵢ - yᵢ    描述统计量     t = d̄/(s_d/√n)
      ↓              ↓              ↓              ↓
  满足前提条件    构建差值序列    检查正态性      df = n - 1
                                                       ↓
                                          查表或计算 p 值 → 做出决策 → 构建置信区间
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 |
|------|------|------|
| $\bar{d}$ | $\sum d_i / n$ | 差值的样本均值 |
| $s_d$ | $\sqrt{\sum(d_i - \bar{d})^2 / (n-1)}$ | 差值的样本标准差 |
| SE | $s_d / \sqrt{n}$ | 差值均值的标准误 |
| $t$ | $\bar{d} / SE$ | 检验统计量 |
| $df$ | $n - 1$ | 自由度 |
| $E$ | $t_{\alpha/2} \times SE$ | 误差范围 |
| $d$ | $|\bar{d}| / s_d$ | Cohen's d 效应量 |

## 12. ❌ 常见误区

### ❌ 误区 1: 把配对数据当作独立样本分析

**✗ 错误做法**: 对同一受试者的前后测量数据使用独立样本 t 检验

**✓ 正确理解**: 配对数据必须使用配对 t 检验。把配对数据当独立样本处理，会丢失个体间的对应关系，导致标准差被高估（因为包含了个体间变异），检验效能大幅降低。在我们的例子中，独立 t 检验的 p 值远大于配对 t 检验。

### ❌ 误区 2: 任何两组数据都可以配对

**✗ 错误做法**: 把两个不相关的样本按顺序「强行配对」

**✓ 正确理解**: 配对必须基于有意义的对应关系——同一受试者的前后测量、匹配的个体（如双胞胎）、同一对象的左右两侧等。如果两组数据之间没有自然的对应关系，强行配对会产生虚假相关，结果毫无意义。

### ❌ 误区 3: 不拒绝 H₀ 就意味着两个方法相同

**✗ 错误做法**: p > 0.05 就下结论「自报体重和实测体重没有区别」

**✓ 正确理解**: 「不拒绝 H₀」只说明没有足够证据证明差异存在，不等于证明差异不存在。可能的原因：(1) 差异确实很小；(2) 样本量不够大，检验效能不足。在我们的 8 对数据中，p = 0.23 但这不代表差异为零——200 对数据的检验就显著了。

### ❌ 误区 4: 配对设计总是比独立设计好

**✗ 错误做法**: 所有情况都优先选择配对设计

**✓ 正确理解**: 配对设计的优势取决于**组内相关性**。当同一受试者的前后测量高度相关时（如体重），配对设计优势明显。但如果配对关系很弱（如随机配对的两个人），配对设计反而会损失自由度（$df = n-1$ 而非 $n_1 + n_2 - 2$）。只有当组内相关 $r > 0.5$ 时，配对设计才比独立设计更有优势。

### ❌ 误区 5: 统计显著就等于实际重要

**✗ 错误做法**: p < 0.05 就认为差异「很重要」

**✓ 正确理解**: 大样本下，即使微不足道的差异也可能统计显著。在我们的 200 对数据中，-3 磅的偏差（不到体重的 2%）高度显著。判断实际重要性需要看**效应量** Cohen's d：小效应（d < 0.2）可能虽然统计显著但实际意义有限。始终同时报告 p 值和效应量。